# 环境安装

本节按照已验证的 Qwen3-1.7B SFT 顺序完成训练前准备。后续环境检查、依赖安装、模型与数据准备以及训练相关命令，均基于该文件执行。

后续实践命令均从当前 Notebook 所在目录出发，并先进入该文件执行。

本节学习大纲如下：

- 通用术语与相关概念
- 准备 Python 3.12 环境
- 检查基础环境
- 创建 Python 虚拟环境并安装依赖
- 安装固定版本的源码项目
- 准备模型和训练数据
- 检查 Python 与 NPU 环境

## 1. 通用术语与相关概念
为便于理解后续内容，学习本课程前请先了解如下术语、缩略语及相关概念。

<table align="left">
<thead><tr><th>术语</th><th>说明</th><th>在本课程中的理解方式</th></tr></thead>
<tbody>
<tr><td>SFT</td><td>Supervised Fine-Tuning，监督微调</td><td>使用有监督样本继续训练已有大模型，使模型适应目标任务或数据</td></tr>
<tr><td>CANN</td><td>Compute Architecture for Neural Networks，昇腾异构计算架构</td><td>Ascend NPU 训练运行所依赖的软件包</td></tr>
<tr><td>ModelScope</td><td>模型与数据资源平台</td><td>本实践用于下载 Qwen3-1.7B 模型</td></tr>
<tr><td>Hugging Face</td><td>模型与数据资源平台</td><td>获取 Wordle 数据集文件</td></tr>
<tr><td>AutoFuse OFF</td><td>AutoFuse 关闭</td><td>作为对照组，观察不开启 AutoFuse 时的训练耗时</td></tr>
<tr><td>AutoFuse ON</td><td>AutoFuse 开启</td><td>作为实验组，与 AutoFuse OFF 的训练耗时进行比较</td></tr>
<tr><td>profiling</td><td>性能采集与分析过程</td><td>用于解释计算、通信等阶段耗时变化</td></tr>
<tr><td>seed</td><td>随机种子</td><td>用于控制随机过程，使同类运行更容易复现</td></tr>
</tbody></table>
<div style="clear:left"></div>

## 2. 准备 Python 3.12 环境

本实践要求使用 Python 3.12 运行训练流程。为了避免影响系统已有 Python 环境，本课程采用独立目录安装 Python 3.12，在当前目录下创建 `env` 目录，用于保存 Python 3.12 安装文件。

执行以下命令完成 Python 3.12 下载、编译和安装：

In [ ]:
%%bash
# 所有新生成文件统一放到当前章节的 tmp 目录
mkdir -p ./tmp
cd ./tmp

# 下载 Python 3.12 源码
curl -O https://mirrors.aliyun.com/python-release/source/Python-3.12.11.tgz
# 解压源码
tar -xf Python-3.12.11.tgz
# 进入源码目录
cd Python-3.12.11
# 指定 Python 安装目录为当前章节的 tmp/python312
./configure --prefix="$(pwd)/../python312"
# 编译 Pytho
make -j32
# 安装 Python 到指定目录
make install

其中：

- `--prefix` 用于指定 Python 安装位置，本实践将 Python 3.12 安装到当前课程目录下的 `tmp/python312`；
- `make -j32` 使用多线程编译，提高编译速度；
- `make install` 将编译后的 Python 安装到指定目录。

安装完成后，检查 Python 版本：

In [ ]:
%%bash
./tmp/python312/bin/python3.12 --version

如果安装成功，应输出 Python 3.12 版本信息。

## 3. 检查基础环境


训练开始前，先确认目标机器 Python 和两张 NPU 是否满足要求。课程中的 `check_env.sh` 位于当前目录下的 `src`。脚本位置保持不变，Python 3.12 则使用前面安装到 `tmp/python312` 中的解释器。

- `CANN_ROOT`：以下使用的是CANN包默认安装路径，以实际指定的安装路径为准，
- `--devices 0,1`：指定检查 0、1 两张 NPU。
- `--python python3.12`：要求使用 Python 3.12。

执行后应重点看输出中的 `[PASS]`。如果出现 `[FAIL]`，应先按照脚本给出的 `action` 修复，不要直接进入训练。

In [ ]:
%%bash
export CANN_ROOT=/home/developer/Ascend/ascend-toolkit/latest
export PYTHON312="$PWD/tmp/python312/bin/python3.12"
bash ./src/check_env.sh --cann-root "$CANN_ROOT" --devices 0,1 --python "$PYTHON312"

## 4. 创建 Python 虚拟环境并安装依赖

这一阶段主要完成两件事：
1. 使用 `python3.12 -m venv` 创建独立 Python 环境，避免和系统已有包相互影响。
2. 安装训练、数据处理所需的固定版本依赖。

`source "$CANN_ROOT/set_env.sh"` 会加载当前 CANN 所需的环境变量，使 Python 侧能够正确找到 Ascend 相关库。

In [ ]:
%%bash
export CANN_ROOT=/home/developer/Ascend/ascend-toolkit/latest

# 使用 tmp 中安装的 Python 3.12 创建虚拟环境，所有依赖均写入 tmp
./tmp/python312/bin/python3.12 -m venv ./tmp/third_party/venv
source ./tmp/third_party/venv/bin/activate
source "$CANN_ROOT/set_env.sh"

python -m pip install --upgrade pip setuptools wheel
python -m pip install \
  torch==2.12.0 torch-npu==2.12.0rc1 triton==3.7.0 \
  torchdata==0.11.0 torchao==0.17.0 datasets==5.0.0 \
  pandas==2.2.3 numpy==1.26.4 scipy==1.13.1 \
  tokenizers==0.23.1 safetensors==0.7.0 fsspec==2026.4.0 \
  tyro==1.0.15 tensorboard==2.20.0 wandb==0.28.1 \
  einops==0.8.2 pillow==12.3.0 PyYAML==6.0.3 \
  modelscope==1.38.1 huggingface-hub==1.24.0 pyarrow==21.0.0
python -m pip install \
  decorator==5.1.1 psutil==6.0.0 loguru==0.7.3 matplotlib==3.11.1 \
  msguard==0.0.8 openpyxl==3.1.5 \
  opentelemetry-exporter-otlp-proto-grpc==1.33.1 \
  opentelemetry-exporter-otlp-proto-http==1.33.1 \
  tzdata==2026.3 plotly==6.9.0 argparse==1.4.0 pybind11==3.0.4 \
  attrs==24.2.0

## 5. 安装固定版本的源码项目
Python 基础依赖安装完成后，还需要准备训练链路中实际使用的源码项目。本实践依赖三个核心组件，它们共同完成从模型训练流程到 Ascend NPU 执行环境的连接。
- `torchtitan`：训练框架主体，负责提供大模型训练所需的基础流程，包括模型加载、数据处理、训练循环以及分布式训练相关能力。本实践基于 `torchtitan` 组织 Qwen3-1.7B 的 SFT（Supervised Fine-Tuning）训练过程。
- `torchtitan-npu`：面向 Ascend NPU 的适配组件。由于原始训练框架主要面向通用 GPU 环境，需要通过 NPU 适配层支持 Ascend 设备上的运行，包括设备调用、训练流程适配以及相关算子支持。
- `inductor_npu_ext`：NPU 侧的 Inductor 扩展组件，用于扩展 PyTorch 编译链路，使模型计算图能够更好地适配 Ascend NPU 后端执行。本实践中的 AutoFuse 能力也依赖完整的编译执行链路，因此需要提前安装该扩展。

In [ ]:
%%bash
export CANN_ROOT=/home/developer/Ascend/ascend-toolkit/latest
source ./tmp/third_party/venv/bin/activate
source "$CANN_ROOT/set_env.sh"

# 创建第三方源码和 wheel 包存放目录，统一放到 tmp 下
# tmp/third_party/src 用于保存源码
# tmp/third_party/wheels 用于保存构建后的安装包
mkdir -p ./tmp/third_party/src ./tmp/third_party/wheels

# 下载训练框架 torchtitan，并固定到验证通过的版本
git clone https://gitcode.com/GitHub_Trending/to/torchtitan.git ./tmp/third_party/src/torchtitan

# 切换到指定 commit，保证训练环境版本一致
git -C ./tmp/third_party/src/torchtitan checkout --detach ac13e536c84e7f6647b14fa9375c3c8a8a2b8578

# 下载 Ascend NPU 适配组件 torchtitan-npu
git clone https://gitcode.com/cann/torchtitan-npu.git ./tmp/third_party/torchtitan-npu

# 固定 torchtitan-npu 版本
git -C ./tmp/third_party/torchtitan-npu checkout --detach 5830760386d590722c4acf694383f4e5c4c0ada1

# 下载 torchair 源码，其中包含 NPU 编译扩展相关代码
git clone https://gitcode.com/Ascend/torchair.git ./tmp/third_party/src/torchair

# 固定 torchair 版本
git -C ./tmp/third_party/src/torchair checkout --detach 3c9418c2804fbb93c2ca5f0c6b9055cbc873f7d1

# 将源码构建成 wheel 安装包
python -m pip wheel --no-deps --wheel-dir ./tmp/third_party/wheels ./tmp/third_party/src/torchtitan
python -m pip wheel --no-deps --wheel-dir ./tmp/third_party/wheels ./tmp/third_party/torchtitan-npu

# 构建 torchair 中的 NPU Inductor 扩展
python -m pip wheel --no-deps --wheel-dir ./tmp/third_party/wheels \
  ./tmp/third_party/src/torchair/experimental/_inductor_npu_ext/python

torchtitan_wheels=(./tmp/third_party/wheels/torchtitan-*.whl)
torchtitan_npu_wheels=(./tmp/third_party/wheels/torchtitan_npu-*.whl)
inductor_wheels=(./tmp/third_party/wheels/inductor_npu_ext-*.whl)

# 检查每个组件是否只生成一个 wheel 文件，如果数量不符合预期，说明构建过程可能失败
[[ ${#torchtitan_wheels[@]} -eq 1 ]] || { printf 'expected one torchtitan wheel\n' >&2; exit 1; }
[[ ${#torchtitan_npu_wheels[@]} -eq 1 ]] || { printf 'expected one torchtitan-npu wheel\n' >&2; exit 1; }
[[ ${#inductor_wheels[@]} -eq 1 ]] || { printf 'expected one inductor-npu-ext wheel\n' >&2; exit 1; }

# 安装刚刚构建的三个 wheel 包
# --force-reinstall 表示覆盖已有版本
python -m pip install --no-deps --force-reinstall \
  "${torchtitan_wheels[0]}" "${torchtitan_npu_wheels[0]}" "${inductor_wheels[0]}"

## 6. 准备模型和训练数据



训练代码准备好后，需要下载真正参与训练的资产：

- Qwen3-1.7B 模型；
- Wordle 数据集中的 parquet 文件。

`modelscope download` 将模型保存到当前章节 `tmp` 目录下的 `assets/hf/Qwen3-1.7B`。


In [ ]:
%%bash
export CANN_ROOT=/home/developer/Ascend/ascend-toolkit/latest
source ./tmp/third_party/venv/bin/activate
source "$CANN_ROOT/set_env.sh"

mkdir -p ./tmp/assets/hf ./tmp/assets/data/wordle

modelscope download --model Qwen/Qwen3-1.7B \
  --local_dir ./tmp/assets/hf/Qwen3-1.7B

export WORDLE_STAGING="$PWD/tmp/assets/.wordle-download"
rm -rf -- "$WORDLE_STAGING"
HF_ENDPOINT=https://hf-mirror.com hf download \
  willcb/V3-wordle data/train-00000-of-00001.parquet --repo-type dataset \
  --local-dir "$WORDLE_STAGING"
install -D -m 0644 \
  "$WORDLE_STAGING/data/train-00000-of-00001.parquet" \
  "$PWD/tmp/assets/data/wordle/train-00000-of-00001.parquet"
rm -rf -- "$WORDLE_STAGING"

## 7. 检查 Python 与 NPU 环境

最后进行一次轻量检查：导入训练所需的关键 Python 包，并通过 `torch.npu.device_count()` 确认当前进程能看到两张 NPU。
`ASCEND_RT_VISIBLE_DEVICES=0,1` 指定本次任务使用 0、1 两张设备。
`env -u LD_PRELOAD -u PYTHONHOME -u CONDA_PREFIX` 用于暂时移除几个容易污染当前 Python/CANN 环境的变量，再启动检查程序。
如果环境正常，最后应打印 `2`，并通过 `assert count == 2`。

In [ ]:
%%bash
export CANN_ROOT=/home/developer/Ascend/ascend-toolkit/latest
source ./tmp/third_party/venv/bin/activate
source "$CANN_ROOT/set_env.sh"
export ASCEND_RT_VISIBLE_DEVICES=0,1
env -u LD_PRELOAD -u PYTHONHOME -u CONDA_PREFIX python - <<'PY'
import torch, torch_npu, torchtitan, torchtitan_npu, triton
import inductor_npu_ext

count = torch.npu.device_count()
print(count)
assert count == 2
PY

## 8. 课后练习

1. （判断题）创建 Python 虚拟环境的主要目的之一，是减少当前实践依赖与系统 Python 环境之间的版本冲突。

2. （单选题）最后的 smoke 检查中，`torch.npu.device_count()` 预期返回多少？
   A. 0
   B. 1
   C. 2
   D. 4

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/04.02_answer.txt